In [5]:
# === Regression Severity ===
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path("D:/PPMI_Project")
df = pd.read_csv(ROOT/"data_processed"/"clinical_with_severity.csv", parse_dates=["INFODT"])

# PURE clinical features (no AGE/SEX)
X_cols = ["NP1TOT","NP2TOT","NP3TOT","MCATOT"]
target = "SEVERITY"

# Clean & impute (safety net)
X = df[X_cols].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median(numeric_only=True))
y = pd.to_numeric(df[target], errors="coerce")
groups = df["PATNO"].astype("Int64")

# --- Keep only rows with a valid target and finite features ---
valid = y.notna()
X, y, groups = X[valid], y[valid], groups[valid]

# ensure finite (no inf/-inf)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True))
mask_fin = np.isfinite(y)
X, y, groups = X[mask_fin], y[mask_fin], groups[mask_fin]

print("Rows after filtering:", len(y))

# Patient-wise split to avoid leakage
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

# XGBoost regressor
model = XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    n_jobs=-1, tree_method="hist", random_state=42
)
model.fit(X_tr, y_tr)

pred = model.predict(X_te)
mae = mean_absolute_error(y_te, pred)
rmse = np.sqrt(mean_squared_error(y_te, pred))
r2 = r2_score(y_te, pred)

print("Test MAE:", round(mae, 4))
print("Test RMSE:", round(rmse, 4))
print("R^2:", round(r2, 4))

# Feature importance (gain)
importances = model.get_booster().get_score(importance_type="gain")
imp_sorted = sorted(importances.items(), key=lambda x: x[1], reverse=True)
print("\nFeature importance (gain):")
for k, v in imp_sorted:
    print(f"{k}: {v:.4f}")

Rows after filtering: 9606
Test MAE: 0.0032
Test RMSE: 0.0052
R^2: 0.9988

Feature importance (gain):
NP2TOT: 0.4256
NP3TOT: 0.2298
NP1TOT: 0.1687
MCATOT: 0.1110


In [7]:
# Next-visit severity forecasting (baseline GBM)
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

ROOT = Path("D:/PPMI_Project")
df = pd.read_csv(ROOT/"data_processed"/"clinical_with_severity.csv", parse_dates=["INFODT"])

# keep only rows with valid severity
df = df[df["SEVERITY"].notna()].copy()

# sort per patient by time
seq = df.sort_values(["PATNO","INFODT"]).copy()

# features (pure clinical + current severity)
X_cols = ["NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY"]
for c in X_cols:
    seq[c] = pd.to_numeric(seq[c], errors="coerce")

# build next-visit label
seq["SEVERITY_NEXT"] = seq.groupby("PATNO")["SEVERITY"].shift(-1)

# drop rows without a next visit
seq = seq.dropna(subset=["SEVERITY_NEXT"]).copy()

# impute features safely
X = seq[X_cols].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
y = pd.to_numeric(seq["SEVERITY_NEXT"], errors="coerce")
groups = seq["PATNO"].astype("Int64")

# filter valid target
valid = y.notna() & np.isfinite(y)
X, y, groups = X[valid], y[valid], groups[valid]

# patient-wise split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))
X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

# model
fcast = XGBRegressor(
    n_estimators=600, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    n_jobs=-1, tree_method="hist", random_state=42
)
fcast.fit(X_tr, y_tr)
pred = fcast.predict(X_te)

# metrics (manual RMSE for older sklearn)
mae = mean_absolute_error(y_te, pred)
rmse = np.sqrt(mean_squared_error(y_te, pred))
r2 = r2_score(y_te, pred)

print("Next-visit Forecast — Test MAE:", round(mae, 4))
print("Next-visit Forecast — Test RMSE:", round(rmse, 4))
print("Next-visit Forecast — R^2:", round(r2, 4))

Next-visit Forecast — Test MAE: 0.0389
Next-visit Forecast — Test RMSE: 0.0585
Next-visit Forecast — R^2: 0.8597


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# Load the enriched file with SEVERITY
# --------------------------------------------------
ROOT = Path("D:/PPMI_Project")
df = pd.read_csv(ROOT/"data_processed"/"clinical_with_severity.csv", parse_dates=["INFODT"])

# --------------------------------------------------
# Helper: attach future severity with time horizon
# --------------------------------------------------
def attach_horizon_target_fast(df, horizon_days, tol_days=90, colname="SEV_FUTURE"):
    out = []
    for pid, g in df.groupby("PATNO"):
        g = g.sort_values("INFODT").copy()

        # convert to numpy date arrays
        dates = g["INFODT"].to_numpy(dtype="datetime64[D]")
        sev   = g["SEVERITY"].to_numpy(dtype=float)
        targets = dates + np.timedelta64(horizon_days, "D")

        # find index of earliest date >= target
        j = np.searchsorted(dates, targets, side="left")

        vals = np.full(len(g), np.nan)
        for i, idx in enumerate(j):
            if idx < len(dates):
                # FIX: convert timedelata to days robustly via division
                diff_days = abs((dates[idx] - targets[i]) / np.timedelta64(1, "D"))
                if diff_days <= tol_days:
                    vals[i] = sev[idx]

        g[colname] = vals
        out.append(g)

    return pd.concat(out, axis=0, ignore_index=True)

# --------------------------------------------------
# Build 6/12/24 Month Targets
# --------------------------------------------------
df = attach_horizon_target_fast(df, 180, tol_days=90,  colname="SEV_6M")
df = attach_horizon_target_fast(df, 365, tol_days=90,  colname="SEV_12M")
df = attach_horizon_target_fast(df, 730, tol_days=120, colname="SEV_24M")

# --------------------------------------------------
# Quick preview
# --------------------------------------------------
print(df[["PATNO","INFODT","SEVERITY","SEV_6M","SEV_12M","SEV_24M"]].head(20))

    PATNO     INFODT  SEVERITY    SEV_6M   SEV_12M  SEV_24M
0    3001 2023-08-01       NaN       NaN  0.651190      NaN
1    3001 2024-09-01  0.651190       NaN       NaN      NaN
2    3001 2024-09-01  0.651190       NaN       NaN      NaN
3    3003 2023-10-01       NaN  0.400566       NaN      NaN
4    3003 2024-05-01  0.400566       NaN  0.284007      NaN
5    3003 2024-05-01  0.400566       NaN  0.284007      NaN
6    3003 2024-09-01  0.388066  0.284007  0.225673      NaN
7    3003 2025-05-01  0.284007       NaN       NaN      NaN
8    3003 2025-09-01  0.225673       NaN       NaN      NaN
9    3004 2024-05-01  0.080933       NaN  0.078162      NaN
10   3004 2024-09-01  0.072600  0.078162  0.078162      NaN
11   3004 2025-05-01  0.078162       NaN       NaN      NaN
12   3004 2025-09-01  0.078162       NaN       NaN      NaN
13   3009 2024-01-01       NaN       NaN       NaN      NaN
14   3009 2024-07-01       NaN       NaN       NaN      NaN
15   3009 2024-12-01       NaN       NaN

In [ ]:
# === Refresh horizons ===
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
P_IN  = ROOT/"data_processed"
P_OUT = ROOT/"data_processed"

src_path = P_IN/"clinical_with_severity.csv"
hor_path = P_OUT/"clinical_with_severity_horizons.csv"

if not src_path.exists():
    raise FileNotFoundError(f"Missing {src_path}. Run earlier steps to create it.")

# ---------------- 1) Load source and CLEAN SEX ----------------
src = pd.read_csv(src_path, parse_dates=["INFODT"])
if "PATNO" not in src.columns:
    raise ValueError("PATNO missing in clinical_with_severity.csv")

# Normalize SEX from common encodings to 1/2 (per your schema: 1=women, 2=men)
def normalize_sex(series: pd.Series) -> pd.Series:
    s = series.copy()
    # string map
    str_map = {
        "f": 1, "female": 1, "woman": 1, "w": 1,
        "m": 2, "male": 2, "man": 2, "h": 2  # sometimes 'H' (hombre) occurs in mixed datasets
    }
    # lower-case strings
    is_str = s.astype(str)
    s_str = is_str.str.strip().str.lower()
    # detect strings that map
    mapped = s_str.map(str_map)
    # numeric where possible
    s_num = pd.to_numeric(series, errors="coerce")
    # prefer numeric if valid (1 or 2), else mapped strings
    out = s_num.where(s_num.isin([1,2]), mapped)
    # anything else remains NaN
    out = out.where(out.isin([1,2]), np.nan)
    return out.astype("Float64")

src["SEX_CLEAN"] = normalize_sex(src.get("SEX"))

# ---------------- 2) Build PATNO→SEX lookup (patient-level mode) ----------------
def mode_ignore_nan(x):
    vals = x.dropna().astype(float)
    if vals.empty: 
        return np.nan
    # if conflicting values exist, take the most frequent; ties → max (2 over 1) purely for determinism
    vc = vals.value_counts()
    top = vc[vc == vc.max()].index.max()
    return float(top)

sex_map = src.groupby("PATNO")["SEX_CLEAN"].apply(mode_ignore_nan).to_dict()

print("Source SEX non-null count:", int(pd.Series(src["SEX_CLEAN"]).notna().sum()))
print("Unique SEX values in source (cleaned):", sorted(pd.Series(src["SEX_CLEAN"].dropna().unique()).astype(int).tolist()))

# ---------------- 3) Load horizons and backfill SEX ----------------
if not hor_path.exists():
    raise FileNotFoundError(f"Missing {hor_path}. Build horizons first.")

hor = pd.read_csv(hor_path, parse_dates=["INFODT"])

before_nonnull = hor["SEX"].notna().sum() if "SEX" in hor.columns else 0
if "SEX" not in hor.columns:
    hor["SEX"] = np.nan

# Clean existing horizons SEX too (in case it holds strings)
hor["SEX"] = normalize_sex(hor["SEX"])

# Fill NaNs from patient lookup
mask_nan = hor["SEX"].isna()
if mask_nan.any():
    hor.loc[mask_nan, "SEX"] = hor.loc[mask_nan, "PATNO"].map(sex_map)

after_nonnull = hor["SEX"].notna().sum()
print(f"Horizons SEX non-null: before={before_nonnull}, after={after_nonnull} (of {len(hor)})")

# Safety: enforce valid values only {1,2}
hor.loc[~hor["SEX"].isin([1,2]), "SEX"] = np.nan

# Quick distribution
vc = hor["SEX"].value_counts(dropna=True).sort_index()
print("SEX distribution in horizons (1=female, 2=male):")
print(vc.to_string())

# ---------------- 4) Save back ----------------
hor.to_csv(hor_path, index=False)
print(f"✅ Updated horizons file with repaired SEX -> {hor_path}")

Source SEX non-null count: 0
Unique SEX values in source (cleaned): []
Horizons SEX non-null: before=0, after=0 (of 13010)
SEX distribution in horizons (1=female, 2=male):
Series([], )
✅ Updated horizons file with repaired SEX -> D:\PPMI_Project\data_processed\clinical_with_severity_horizons.csv


In [ ]:
# === Auto-discover SEX from other CSVs and backfill into horizons ===
import pandas as pd, numpy as np, os, gc
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
P_IN  = ROOT/"data_processed"
P_OUT = ROOT/"data_processed"
SEARCH_DIRS = [ROOT, ROOT/"data_raw", ROOT/"data", P_IN, P_OUT]  # add/adjust if your data lives elsewhere

hor_path = P_OUT/"clinical_with_severity_horizons.csv"
if not hor_path.exists():
    raise FileNotFoundError(f"Missing {hor_path}. Build horizons first.")

# --- helpers ---
def normalize_sex(series: pd.Series) -> pd.Series:
    if series is None:
        return pd.Series(dtype="Float64")
    s = series.copy()
    # map strings
    str_map = {
        "f":1, "female":1, "woman":1, "w":1,
        "m":2, "male":2, "man":2, "h":2
    }
    s_str = s.astype(str).str.strip().str.lower()
    mapped = s_str.map(str_map)
    s_num = pd.to_numeric(s, errors="coerce")
    out = s_num.where(s_num.isin([1,2]), mapped)
    out = out.where(out.isin([1,2]), np.nan)
    return out.astype("Float64")

CAND_COLS = {
    # likely exact
    "SEX", "Sex", "sex",
    "GENDER", "Gender", "gender",
    # sometimes appear in PPMI exports/joins
    "PAGENDER", "PA_GENDER", "PAT_GENDER", "SEXNUM", "SEX_NUM", "SEX_CODE", "SEXCODE"
}

def find_candidate_files():
    seen = set()
    files = []
    for d in SEARCH_DIRS:
        if not d.exists(): 
            continue
        for p in d.rglob("*.csv"):
            # de-duplicate by absolute path
            ap = p.resolve()
            if ap in seen: 
                continue
            seen.add(ap)
            files.append(ap)
    # likely demographics first
    files.sort(key=lambda p: (0 if "demo" in p.name.lower() or "gender" in p.name.lower() else 1, str(p)))
    return files

def collect_sex_map():
    best_map = None
    best_nonnull = 0
    best_file = None
    for p in find_candidate_files():
        # quick light read of header
        try:
            df_head = pd.read_csv(p, nrows=5)
        except Exception:
            continue
        cols = set(df_head.columns.astype(str))
        if "PATNO" not in cols:
            continue
        sex_col = None
        # try exact hits first
        for c in cols:
            if c in CAND_COLS:
                sex_col = c
                break
        # fallback: fuzzy search in col names
        if sex_col is None:
            for c in cols:
                lc = c.lower()
                if ("sex" == lc) or ("gender" == lc) or (lc.endswith("_sex")) or (lc.endswith("_gender")):
                    sex_col = c
                    break
        if sex_col is None:
            continue

        # read only necessary columns
        try:
            df = pd.read_csv(p, usecols=["PATNO", sex_col])
        except Exception:
            # some files have spaces or odd encodings; try engine python
            try:
                df = pd.read_csv(p, usecols=["PATNO", sex_col], engine="python")
            except Exception:
                continue

        df["SEX_CLEAN"] = normalize_sex(df[sex_col])
        grp = df.groupby("PATNO")["SEX_CLEAN"].agg(lambda x: x.dropna().mode().max() if x.dropna().size>0 else np.nan)
        nonnull = grp.notna().sum()

        if nonnull > best_nonnull:
            best_nonnull = nonnull
            best_map = grp
            best_file = p

        del df, df_head
        gc.collect()

    return best_map, best_nonnull, best_file

# --- do it ---
sex_map, nonnull, src_file = find_candidate_files() and collect_sex_map()
if sex_map is None or nonnull == 0:
    raise RuntimeError(
        "Could not find any CSV with PATNO and a usable SEX/Gender column. "
        "Please place a Demographics-like CSV (with PATNO and Gender/SEX) in D:/PPMI_Project (or subfolders) and rerun."
    )

print(f"✅ Found SEX in: {src_file}")
print(f"Usable PATNO→SEX entries: {nonnull}")

# load horizons and backfill
hor = pd.read_csv(hor_path, parse_dates=["INFODT"])
if "SEX" not in hor.columns:
    hor["SEX"] = np.nan
hor["SEX"] = normalize_sex(hor["SEX"])

before = int(hor["SEX"].notna().sum())
hor.loc[hor["SEX"].isna(), "SEX"] = hor.loc[hor["SEX"].isna(), "PATNO"].map(sex_map)
# enforce valid values
hor.loc[~hor["SEX"].isin([1,2]), "SEX"] = np.nan
after = int(hor["SEX"].notna().sum())

print(f"Horizons SEX non-null: before={before}, after={after}, total={len(hor)}")
print("Distribution after fill (1=female, 2=male):")
print(hor["SEX"].value_counts(dropna=True).sort_index().to_string())

hor.to_csv(hor_path, index=False)
print(f"✅ Updated horizons file -> {hor_path}")

✅ Found SEX in: D:\PPMI_Project\data_raw\clinical\Demographics.csv
Usable PATNO→SEX entries: 3749
Horizons SEX non-null: before=0, after=7332, total=13010
Distribution after fill (1=female, 2=male):
SEX
1.0    7332
✅ Updated horizons file -> D:\PPMI_Project\data_processed\clinical_with_severity_horizons.csv


In [ ]:
# === Diagnose & correct SEX mapping (Demographics -> Horizons) ===
import pandas as pd, numpy as np
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
DEMO_CSV = ROOT/"data_raw/clinical/Demographics.csv"
HOR_CSV  = ROOT/"data_processed/clinical_with_severity_horizons.csv"

def find_sex_col(df):
    cand = [c for c in df.columns if c.lower() in {"sex","gender"}]
    if cand: return cand[0]
    # fuzzy
    for c in df.columns:
        lc = c.lower()
        if lc.endswith("_sex") or lc.endswith("_gender") or "sex" in lc or "gender" in lc:
            return c
    raise ValueError("Could not find a SEX/Gender column in Demographics.csv")

def normalize_try(df, col, scheme="strings_then_12_female1_male2"):
    s = df[col]
    # string mapping first
    s_str = s.astype(str).str.strip().str.lower()
    str_map = {"f":1,"female":1,"woman":1,"w":1,
               "m":2,"male":2,"man":2,"h":2}
    mapped = s_str.map(str_map)

    # numeric candidates
    num = pd.to_numeric(s, errors="coerce")

    out = None
    if scheme == "strings_then_12_female1_male2":
        out = num.where(num.isin([1,2]), mapped)
        # enforce only 1/2
        out = out.where(out.isin([1,2]), np.nan)
    elif scheme == "strings_then_12_male1_female2":
        # flip 1/2 meaning
        flip = num.map({1:2, 2:1})
        out = flip.where(flip.isin([1,2]), mapped)
        out = out.where(out.isin([1,2]), np.nan)
    elif scheme == "strings_then_01_female1_male0":
        # treat 0/1 as male/female -> map to 2/1
        m01 = num.map({0:2, 1:1})
        out = m01.where(m01.isin([1,2]), mapped)
        out = out.where(out.isin([1,2]), np.nan)
    else:
        raise ValueError("unknown scheme")
    return out.astype("Float64")

def pick_best_mapping(df, col):
    schemes = [
        "strings_then_12_female1_male2",
        "strings_then_12_male1_female2",
        "strings_then_01_female1_male0",
    ]
    scores = []
    for sch in schemes:
        se = normalize_try(df, col, sch)
        vc = se.value_counts(dropna=True)
        n1 = int(vc.get(1.0, 0))  # female
        n2 = int(vc.get(2.0, 0))  # male
        total = n1 + n2
        if total == 0:
            score = -1
        else:
            # prefer mappings with both sexes present and reasonable male fraction (0.35–0.75)
            male_frac = n2 / total
            penalty = 0
            if n1 == 0 or n2 == 0: penalty += 2
            penalty += abs(male_frac - 0.6)  # PD cohorts often ~60% male
            score = 10 - penalty
        scores.append((score, sch, n1, n2))
    scores.sort(reverse=True)
    best = scores[0]
    return best  # (score, scheme, n_female, n_male)

# 1) Load demographics & detect column
demo = pd.read_csv(DEMO_CSV)
sex_col = find_sex_col(demo)
print(f"Detected SEX column: {sex_col}")
print("Raw value counts (top 10):")
print(demo[sex_col].value_counts(dropna=False).head(10).to_string())

# 2) Choose best mapping automatically
score, scheme, nF, nM = pick_best_mapping(demo, sex_col)
sex_series = normalize_try(demo, sex_col, scheme)
print(f"\nChosen mapping scheme: {scheme}")
print(f"Mapped counts -> female(1): {nF}, male(2): {nM}")

# Build PATNO->SEX map (patient-level mode)
if "PATNO" not in demo.columns:
    raise ValueError("PATNO missing in demographics.")
demo["_SEX_CLEAN_"] = sex_series
sex_map = demo.groupby("PATNO")["_SEX_CLEAN_"].agg(lambda x: x.dropna().mode().max() if x.dropna().size>0 else np.nan)

# 3) Backfill horizons
hor = pd.read_csv(HOR_CSV, parse_dates=["INFODT"])
if "SEX" not in hor.columns:
    hor["SEX"] = np.nan

def enforce(x):
    x = pd.to_numeric(x, errors="coerce")
    x = x.where(x.isin([1,2]), np.nan)
    return x.astype("Float64")

hor["SEX"] = enforce(hor["SEX"])
before = int(hor["SEX"].notna().sum())
mask = hor["SEX"].isna()
hor.loc[mask, "SEX"] = hor.loc[mask, "PATNO"].map(sex_map)
hor["SEX"] = enforce(hor["SEX"])
after = int(hor["SEX"].notna().sum())

print(f"\nHorizons SEX non-null: before={before}, after={after}, total={len(hor)}")
print("Distribution after fix (1=female, 2=male):")
print(hor["SEX"].value_counts(dropna=True).sort_index().to_string())

hor.to_csv(HOR_CSV, index=False)
print(f"✅ Updated horizons file -> {HOR_CSV}")

Detected SEX column: SEX
Raw value counts (top 10):
SEX
0.0    3866
1.0    3749
NaN       2

Chosen mapping scheme: strings_then_01_female1_male0
Mapped counts -> female(1): 3749, male(2): 3866

Horizons SEX non-null: before=7332, after=13010, total=13010
Distribution after fix (1=female, 2=male):
SEX
1.0    7332
2.0    5678
✅ Updated horizons file -> D:\PPMI_Project\data_processed\clinical_with_severity_horizons.csv


In [ ]:
# === Build horizons if missing, then train & save models ===
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import joblib

ROOT = Path("D:/PPMI_Project")
P_IN  = ROOT/"data_processed"
P_OUT = ROOT/"data_processed"
P_MOD = ROOT/"models"
P_MOD.mkdir(parents=True, exist_ok=True)

horizons_path = P_OUT/"clinical_with_severity_horizons.csv"

# ---------- If horizons file is missing, build it from clinical_with_severity.csv ----------
if not horizons_path.exists():
    src = P_IN/"clinical_with_severity.csv"
    if not src.exists():
        raise FileNotFoundError(f"Missing {src}. Run the severity step (Cell A) first.")
    df = pd.read_csv(src, parse_dates=["INFODT"])
    # keep expected columns if present
    keep = [c for c in ["PATNO","EVENT_ID","INFODT","AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY","Y_PD"] if c in df.columns]
    df = df[keep].copy()
    # types
    df["PATNO"] = pd.to_numeric(df["PATNO"], errors="coerce").astype("Int64")
    for c in ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.sort_values(["PATNO","INFODT"])

    # fast horizon attachment (find nearest future visit around target date)
    def attach_horizon_target_fast(df, horizon_days, tol_days, colname):
        out = []
        for pid, g in df.groupby("PATNO", sort=False):
            g = g.sort_values("INFODT").copy()
            dates = g["INFODT"].to_numpy()
            sev   = g["SEVERITY"].to_numpy()
            targets = dates + np.timedelta64(horizon_days, "D")
            idx = np.searchsorted(dates, targets, side="left")
            vals = np.full(len(g), np.nan, dtype=float)
            for i, j in enumerate(idx):
                candidates = []
                # candidate j (>= target)
                if j < len(dates):
                    candidates.append((j, abs((dates[j] - targets[i]).astype("timedelta64[D]").astype(int))))
                # candidate j-1 (< target)
                if j-1 >= 0:
                    candidates.append((j-1, abs((dates[j-1] - targets[i]).astype("timedelta64[D]").astype(int))))
                if candidates:
                    k, d = min(candidates, key=lambda x: x[1])
                    if d <= tol_days and not pd.isna(sev[k]):
                        vals[i] = sev[k]
            g[colname] = vals
            out.append(g)
        return pd.concat(out, axis=0, ignore_index=True)

    df = attach_horizon_target_fast(df, 180, tol_days=90,  colname="SEV_6M")
    df = attach_horizon_target_fast(df, 365, tol_days=90,  colname="SEV_12M")
    df = attach_horizon_target_fast(df, 730, tol_days=120, colname="SEV_24M")

    df.to_csv(horizons_path, index=False)
    print(f"✅ Built horizons file -> {horizons_path}")

# ---------- Train & save horizon models ----------
df = pd.read_csv(horizons_path, parse_dates=["INFODT"])
for c in ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY","SEV_6M","SEV_12M","SEV_24M"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

print("Non-null horizon counts:")
for c in ["SEV_6M","SEV_12M","SEV_24M"]:
    print(f"  {c}: {df[c].notna().sum()} of {len(df)}")

base_feats = ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY"]
print("\nUsing features:", base_feats)

def prepare_xy(data, target_col):
    use = data[data[target_col].notna()].copy()
    feat_cols = [f for f in base_feats if f in use.columns]
    drop_all_nan = [f for f in feat_cols if use[f].isna().all()]
    if drop_all_nan:
        print(f"   dropping all-NaN columns for {target_col}: {drop_all_nan}")
        feat_cols = [f for f in feat_cols if f not in drop_all_nan]
    X = use[feat_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    y = pd.to_numeric(use[target_col], errors="coerce")
    groups = use["PATNO"].astype("Int64") if "PATNO" in use.columns else pd.Series(np.arange(len(use)))
    return use, X, y, groups, feat_cols

def train_eval(target_col, stem):
    use, X, y, groups, feat_cols = prepare_xy(df, target_col)
    print(f"\n=== {target_col} ===")
    print("rows with target:", len(use))
    if len(use) < 200:
        print("⚠️  Too few rows — skipping.")
        return None, (np.nan, np.nan, np.nan)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    tr_idx, te_idx = next(gss.split(X, y, groups=groups))
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

    model = XGBRegressor(
        n_estimators=600, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        n_jobs=-1, tree_method="hist", random_state=42
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    mae  = mean_absolute_error(y_te, pred)
    rmse = mean_squared_error(y_te, pred) ** 0.5  
    r2   = r2_score(y_te, pred)
    print("usable rows:", len(use))
    print("Test MAE :", round(mae, 4))
    print("Test RMSE:", round(rmse, 4))
    print("R^2      :", round(r2, 4))

    # save
    model.get_booster().save_model(P_MOD/f"{stem}.json")
    joblib.dump(model, P_MOD/f"{stem}.pkl")
    print(f"✅ Saved -> {stem}.json & {stem}.pkl")
    return model, (mae, rmse, r2)

summary = {}
for tgt, stem in [("SEV_6M","severity_6m"), ("SEV_12M","severity_12m"), ("SEV_24M","severity_24m")]:
    m, met = train_eval(tgt, stem)
    summary[tgt] = met

print("\nSummary:")
for k,v in summary.items():
    print(f"{k}: MAE={v[0]}, RMSE={v[1]}, R2={v[2]}")

Non-null horizon counts:
  SEV_6M: 8074 of 13010
  SEV_12M: 5099 of 13010
  SEV_24M: 747 of 13010

Using features: ['AGE', 'SEX', 'NP1TOT', 'NP2TOT', 'NP3TOT', 'MCATOT', 'SEVERITY']

=== SEV_6M ===
rows with target: 8074
usable rows: 8074
Test MAE : 0.0508
Test RMSE: 0.0737
R^2      : 0.7844
✅ Saved -> severity_6m.json & severity_6m.pkl

=== SEV_12M ===
rows with target: 5099
usable rows: 5099
Test MAE : 0.0667
Test RMSE: 0.0918
R^2      : 0.6655
✅ Saved -> severity_12m.json & severity_12m.pkl

=== SEV_24M ===
rows with target: 747
usable rows: 747
Test MAE : 0.0755
Test RMSE: 0.1031
R^2      : 0.6901
✅ Saved -> severity_24m.json & severity_24m.pkl

Summary:
SEV_6M: MAE=0.05081038298881819, RMSE=0.07373581955736933, R2=0.7844265545994094
SEV_12M: MAE=0.0666865033998219, RMSE=0.09179848742761122, R2=0.6654913727095089
SEV_24M: MAE=0.07545622760702382, RMSE=0.10307053829911167, R2=0.6901113376249867


In [ ]:
# ===  Robust eval (GroupKFold) + diagnostics + SHAP exports ===
import json, math, gc
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import joblib

ROOT = Path("D:/PPMI_Project")
P_OUT = ROOT/"data_processed"
P_MOD = ROOT/"models"
P_EVAL = ROOT/"eval"
P_INT  = ROOT/"interpretability"
for p in [P_EVAL, P_INT]: p.mkdir(parents=True, exist_ok=True)

horizons_path = P_OUT/"clinical_with_severity_horizons.csv"
df = pd.read_csv(horizons_path, parse_dates=["INFODT"])

cols_num = ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY","SEV_6M","SEV_12M","SEV_24M"]
for c in cols_num:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

BASE_FEATS = ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY"]

def prepare_xy(data, target_col):
    use = data[data[target_col].notna()].copy()
    feats = [f for f in BASE_FEATS if f in use.columns and not use[f].isna().all()]
    X = use[feats].apply(pd.to_numeric, errors="coerce").replace([np.inf,-np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))
    y = pd.to_numeric(use[target_col], errors="coerce")
    groups = use["PATNO"].astype("Int64") if "PATNO" in use.columns else pd.Series(np.arange(len(use)))
    return use, X, y, groups, feats

def groupk_eval(model_pkl, X, y, groups, k=5):
    from xgboost import XGBRegressor
    base = joblib.load(model_pkl)
    params = base.get_xgb_params()
    params.update(dict(n_estimators=base.n_estimators, tree_method="hist", random_state=42))
    gkf = GroupKFold(n_splits=min(k, len(np.unique(groups))))
    rows = []; y_all=[]; p_all=[]; g_all=[]
    for fi,(tr,te) in enumerate(gkf.split(X,y,groups)):
        m = XGBRegressor(**params)
        m.fit(X.iloc[tr], y.iloc[tr])
        pred = m.predict(X.iloc[te])
        yt = y.iloc[te].to_numpy()
        rows.append(dict(fold=fi+1, n_test=len(te),
                         MAE=mean_absolute_error(yt,pred),
                         RMSE=float(np.sqrt(((pred - yt) ** 2).mean())),
                         R2=r2_score(yt,pred)))
        y_all.append(yt); p_all.append(pred); g_all.append(groups.iloc[te].to_numpy())
    y_all=np.concatenate(y_all); p_all=np.concatenate(p_all); g_all=np.concatenate(g_all)
    agg = dict(MAE=float(np.mean([r["MAE"] for r in rows])),
               RMSE=float(np.mean([r["RMSE"] for r in rows])),
               R2=float(np.mean([r["R2"] for r in rows])),
               N=len(y_all))
    return pd.DataFrame(rows), agg, y_all, p_all, g_all

def diag_plots(tag, y_true, y_pred):
    # Parity
    lim_min=float(np.nanmin([y_true,y_pred])); lim_max=float(np.nanmax([y_true,y_pred]))
    plt.figure(figsize=(5,5))
    plt.scatter(y_true,y_pred,s=6,alpha=0.4)
    plt.plot([lim_min,lim_max],[lim_min,lim_max],'--'); plt.xlabel("True"); plt.ylabel("Predicted")
    plt.title(f"{tag} — Parity"); plt.tight_layout()
    plt.savefig(P_EVAL/f"{tag}_parity.png",dpi=180); plt.close()

    # Residuals
    resid = y_pred - y_true
    plt.figure(figsize=(6,4))
    plt.hist(resid,bins=40); plt.xlabel("Residual (Pred-True)"); plt.ylabel("Count")
    plt.title(f"{tag} — Residuals"); plt.tight_layout()
    plt.savefig(P_EVAL/f"{tag}_residuals.png",dpi=180); plt.close()

    # Calibration (bin means)
    bins = np.linspace(0,1,11); idx = np.digitize(y_pred,bins)-1
    dfc = pd.DataFrame({"y_true":y_true,"y_pred":y_pred,"bin":idx})
    cal = dfc.groupby("bin").agg(y_true_mean=("y_true","mean"),
                                 y_pred_mean=("y_pred","mean"),
                                 n=("y_true","size")).reset_index()
    plt.figure(figsize=(5,5))
    plt.plot([0,1],[0,1],"--")
    plt.scatter(cal["y_pred_mean"],cal["y_true_mean"],s=np.clip(cal["n"],10,120))
    plt.xlabel("Predicted (bin mean)"); plt.ylabel("True (bin mean)")
    plt.title(f"{tag} — Calibration"); plt.tight_layout()
    plt.savefig(P_EVAL/f"{tag}_calibration.png",dpi=180); plt.close()

def safe_shap_export(stem, X, feats):
    """Creates beeswarm, bar top-15, and CSV; skips gracefully if SHAP not installed."""
    try:
        import shap
    except Exception as e:
        print(f"SHAP not available, skipping ({e})")
        return
    model = joblib.load(P_MOD/f"{stem}.pkl")
    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X, check_additivity=False)
    except Exception as e:
        print(f"SHAP explainer error: {e}")
        return
    # Beeswarm
    plt.figure(figsize=(8,5)); shap.summary_plot(shap_values, X, show=False)
    plt.tight_layout(); plt.savefig(P_INT/f"shap_{stem}_beeswarm.png", dpi=180); plt.close()
    # Bar top-15
    plt.figure(figsize=(8,5)); shap.summary_plot(shap_values, X, plot_type="bar", max_display=15, show=False)
    plt.tight_layout(); plt.savefig(P_INT/f"shap_{stem}_bar.png", dpi=180); plt.close()
    # Table
    imp = np.abs(shap_values).mean(axis=0)
    order = np.argsort(-imp)
    top = pd.DataFrame({"feature": np.array(feats)[order][:15],
                        "mean_abs_shap": imp[order][:15]})
    top.to_csv(P_INT/f"shap_{stem}_top15.csv", index=False)
    print(f"✅ SHAP saved for {stem}")

summary_rows = []

for tgt, stem in [("SEV_6M","severity_6m"),
                  ("SEV_12M","severity_12m"),
                  ("SEV_24M","severity_24m")]:

    use, X, y, groups, feats = prepare_xy(df, tgt)
    if len(use) < 200:
        print(f"⚠️ {tgt}: too few rows for robust CV — skipping.")
        continue

    # Save features used
    with open(P_EVAL/f"{stem}_features.json","w") as f:
        json.dump({"features": feats}, f, indent=2)

    folds_df, agg, y_all, p_all, g_all = groupk_eval(P_MOD/f"{stem}.pkl", X, y, groups, k=5)
    folds_df.to_csv(P_EVAL/f"{stem}_folds.csv", index=False)
    pd.DataFrame({"PATNO": g_all, "y_true": y_all, "y_pred": p_all}).to_csv(P_EVAL/f"{stem}_predictions.csv", index=False)
    diag_plots(tgt, y_all, p_all)

    summary_rows.append(dict(Horizon=tgt, **agg))
    # SHAP on a stratified subset to keep it fast if huge
    n_sample = min(4000, len(X))
    X_sample = X.sample(n=n_sample, random_state=42) if len(X) > n_sample else X
    safe_shap_export(stem, X_sample, feats)
    del X_sample; gc.collect()

# Global summary
if summary_rows:
    eval_summary = pd.DataFrame(summary_rows).sort_values("Horizon")
    eval_summary.to_csv(P_EVAL/"eval_summary.csv", index=False)
    print("✅ Wrote:", P_EVAL/"eval_summary.csv")
    print(eval_summary)

✅ SHAP saved for severity_6m
✅ SHAP saved for severity_12m
✅ SHAP saved for severity_24m
✅ Wrote: D:\PPMI_Project\eval\eval_summary.csv
   Horizon       MAE      RMSE        R2     N
1  SEV_12M  0.063538  0.086587  0.702768  5099
2  SEV_24M  0.081752  0.110047  0.548826   747
0   SEV_6M  0.049799  0.071320  0.786378  8074


In [ ]:
# === Model cards + versions ===
import sys, platform, json, joblib, xgboost
from datetime import datetime
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
P_MOD = ROOT/"models"
P_EVAL = ROOT/"eval"
P_MOD.mkdir(exist_ok=True, parents=True)

def snapshot_one(stem, target):
    m = joblib.load(P_MOD/f"{stem}.pkl")
    card = {
        "model_name": stem,
        "target": target,
        "created_utc": datetime.utcnow().isoformat()+"Z",
        "framework": "XGBRegressor",
        "xgboost_version": xgboost.__version__,
        "python": sys.version,
        "platform": platform.platform(),
        "params": m.get_xgb_params(),
        "n_estimators": getattr(m, "n_estimators", None),
        "features_file": f"../eval/{stem}_features.json",
        "eval_summary_file": "../eval/eval_summary.csv",
        "notes": "Group-wise CV by PATNO; severity scale ~[0,1]. SEX fixed from Demographics."
    }
    with open(P_MOD/f"{stem}.modelcard.json","w") as f:
        json.dump(card, f, indent=2)
    return card

cards = []
for tgt, stem in [("SEV_6M","severity_6m"),
                  ("SEV_12M","severity_12m"),
                  ("SEV_24M","severity_24m")]:
    try:
        cards.append(snapshot_one(stem, tgt))
        print("✅ wrote", stem+".modelcard.json")
    except Exception as e:
        print("skip", stem, e)

# Minimal freeze (pin xgboost; others can be left floating or pinned if you prefer)
with open(P_MOD/"requirements_freeze.txt","w") as f:
    f.write("\n".join([
        f"xgboost=={xgboost.__version__}",
        "scikit-learn",
        "pandas",
        "numpy",
        "joblib",
        "matplotlib",
        "shap"
    ]))
print("✅ wrote requirements_freeze.txt")

✅ wrote severity_6m.modelcard.json
✅ wrote severity_12m.modelcard.json
✅ wrote severity_24m.modelcard.json
✅ wrote requirements_freeze.txt


In [ ]:
# === Inference wrappers (raw + optional isotonic calibration) ===
import joblib, numpy as np, pandas as pd
from pathlib import Path

ROOT = Path("D:/PPMI_Project")
P_MOD = ROOT/"models"
P_OUT = ROOT/"data_processed"

BASE_FEATS = ["AGE","SEX","NP1TOT","NP2TOT","NP3TOT","MCATOT","SEVERITY"]
HORIZONS   = [("SEV_6M","severity_6m"),
              ("SEV_12M","severity_12m"),
              ("SEV_24M","severity_24m")]

# Lazy caches
__MODEL_CACHE = {}
__ISO_CACHE   = {}

def _load_model(stem):
    if stem not in __MODEL_CACHE:
        __MODEL_CACHE[stem] = joblib.load(P_MOD/f"{stem}.pkl")
    return __MODEL_CACHE[stem]

def _maybe_load_iso(stem):
    """Load isotonic if available (created by the optional calibration step)."""
    if stem in __ISO_CACHE:
        return __ISO_CACHE[stem]
    iso_path = P_MOD/f"{stem}.iso.pkl"
    if iso_path.exists():
        __ISO_CACHE[stem] = joblib.load(iso_path)
    else:
        __ISO_CACHE[stem] = None
    return __ISO_CACHE[stem]

def _prep_X(df_like: pd.DataFrame):
    X = df_like.copy()
    cols = [c for c in BASE_FEATS if c in X.columns]
    # drop all-NaN features gracefully
    cols = [c for c in cols if not X[c].isna().all()]
    X = X[cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    med = X.median(numeric_only=True)
    X = X.fillna(med)
    return X, cols

def predict_horizons(input_row_or_df: pd.DataFrame,
                     calibrated: bool = True,
                     clip01: bool = True):
    """
    Predict SEV_6M, SEV_12M, SEV_24M for a single row or batch.
    Returns dict: {"used_features": [...], "SEV_6M": np.array, ...,
                   "SEV_6M_cal": np.array (if calibrated=True and iso available), ...}
    """
    if isinstance(input_row_or_df, pd.Series):
        df_in = input_row_or_df.to_frame().T
    else:
        df_in = input_row_or_df.copy()

    X, used_cols = _prep_X(df_in)
    out = {"used_features": used_cols}

    for tgt, stem in HORIZONS:
        m = _load_model(stem)
        pred = m.predict(X).astype(float)
        if clip01:
            pred = np.clip(pred, 0.0, 1.0)
        out[tgt] = pred

        if calibrated:
            iso = _maybe_load_iso(stem)
            if iso is not None:
                pred_cal = iso.transform(pred)
                if clip01:
                    pred_cal = np.clip(pred_cal, 0.0, 1.0)
                out[f"{tgt}_cal"] = pred_cal

    return out

# Convenience: predict on latest visit per patient (requires PATNO/INFODT in CSV)
def predict_latest_per_patient(horizons_csv=ROOT/"data_processed/clinical_with_severity_horizons.csv",
                               calibrated: bool = True,
                               clip01: bool = True):
    df = pd.read_csv(horizons_csv, parse_dates=["INFODT"])
    if not {"PATNO","INFODT"}.issubset(df.columns):
        raise ValueError("predict_latest_per_patient() needs PATNO and INFODT in the CSV.")
    df = df.sort_values(["PATNO","INFODT"])
    last = df.groupby("PATNO", as_index=False).tail(1)
    needed = [c for c in BASE_FEATS if c in last.columns]
    preds = predict_horizons(last[needed], calibrated=calibrated, clip01=clip01)
    out = last[["PATNO","INFODT"]].reset_index(drop=True)
    for k in [h[0] for h in HORIZONS]:
        out[k] = preds[k]
        if f"{k}_cal" in preds:
            out[f"{k}_cal"] = preds[f"{k}_cal"]
    return out

# --- Examples (commented) ---
# Single row:
# row = pd.DataFrame([{
#   "AGE": 66.0, "SEX": 2, "NP1TOT": 3, "NP2TOT": 9, "NP3TOT": 22, "MCATOT": 2, "SEVERITY": 0.42
# }])
# print(predict_horizons(row))

# Batch: latest per patient and save
# latest = predict_latest_per_patient()
# latest.to_csv(ROOT/"eval/pred_latest_per_patient.csv", index=False)
# print("✅ wrote", ROOT/"eval/pred_latest_per_patient.csv")